<a href="https://colab.research.google.com/github/RichardEager0803/GenAI/blob/main/HW4/HW4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework #4

Set up the environment and gathered six historical documents to run the experiment.

In [ ]:
!pip install -qU langchain-core langchain-community langchain-google-genai langchain-classic
!pip install -qU faiss-cpu pypdf rank_bm25 flashrank

# Standard imports for environment management
import os
from google.colab import userdata

# API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


Chunking step with a total of 100 chunks.

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

file_names = [
    "four_freedoms.txt",
    "i_have_a_dream.txt",
    "day_of_infamy.txt",
    "second_inaugural.txt",
    "a_time_for_choosing.txt",
    "first_inaugural.txt"
]

all_docs = []

# Loop through each file and split it into chunks
for file in file_names:
    if os.path.exists(file):
        print(f"Processing {file}...")

        loader = TextLoader(file, encoding="utf-8")

        # Recursive splitter attempts to keep paragraphs and sentences together
        # chunk_size is total characters; overlap keeps context across boundaries
        splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
        chunks = loader.load_and_split(splitter)

        all_docs.extend(chunks)
    else:
        print(f"Warning: '{file}' not found in the sidebar. Please upload it.")

if all_docs:
    print(f"\nSuccess! Total text chunks created: {len(all_docs)}")

Processing four_freedoms.txt...
Processing i_have_a_dream.txt...
Processing day_of_infamy.txt...
Processing second_inaugural.txt...
Processing a_time_for_choosing.txt...
Processing first_inaugural.txt...

Success! Total text chunks created: 100


# Setting up RAG Retrievers for Experiments
*   To evaluate different retrieval strategies, I set up three different retrievers.

# Trial A: Naive Vector Similarity
*   Uses FAISS vector embeddings (Gemini) to find the chunks that are closest in meaning to the query, may retrieve redundant chunks.


# Trial B: MMR (Maximal Marginal Relevance)
*   Uses vector embeddings but balances the relevance and the diversity, fetch_k=20 creates a candidate pool, and k=5 selects the final chunks

# Trial C: Hybrid Retriever (Vector + BM25)
*   Combines semantic search and keyword search to give the best of both worlds. Semantic finds text based meaning (not exact words), while keyword uses a traditional lexical index.




In [5]:
import time
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Initialize Gemini Embeddings (2026 Stable Model)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)

# Define the Retry-Safe Embedding Wrapper
# This function will wait and retry automatically if we get a 429 error.
@retry(
    wait=wait_random_exponential(min=1, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(Exception) # Catch the GoogleGenerativeAIError
)
def embed_with_retry(vector_store, batch):
    if vector_store is None:
        return FAISS.from_documents(batch, embeddings)
    else:
        vector_store.add_documents(batch)
        return vector_store

# Process with Batching and Retries
batch_size = 15 # Smaller batches help prevent hitting the limit too fast
vectorstore = None

print(f"Embedding {len(all_docs)} chunks with rate-limit protection...")
for i in range(0, len(all_docs), batch_size):
    batch = all_docs[i : i + batch_size]
    try:
        vectorstore = embed_with_retry(vectorstore, batch)
        print(f" Processed {i + len(batch)}/{len(all_docs)}...")
    except Exception as e:
        print(f" Failed after retries: {e}")
        break
    time.sleep(3) # A steady 3-second heartbeat to keep the API happy

# Finalize Retrievers
if vectorstore:
    # Trial A: Naive Vector Similarity
    naive_vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    # Trial B: MMR Retriever for diversity
    mmr_retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
          "k": 5,        # final top chunks
          "fetch_k": 20  # initial candidates for diversity
        }
    )

    # Trial C: Hybrid Retriever (Vector + BM25)
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    bm25_retriever = BM25Retriever.from_documents(all_docs)
    bm25_retriever.k = 5

    hybrid_retriever = EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[0.5, 0.5]
    )

    print("\nAll retrievers are ready!")
    print("Trial A: Naive Vector")
    print("Trial B: MMR")
    print("Trial C: Hybrid (Vector + BM25)")



Embedding 100 chunks with rate-limit protection...
 Processed 15/100...
 Processed 30/100...
 Processed 45/100...
 Processed 60/100...
 Processed 75/100...
 Processed 90/100...
 Processed 100/100...

All retrievers are ready!
Trial A: Naive Vector
Trial B: MMR
Trial C: Hybrid (Vector + BM25)


Testing chunk retrieval with Naive Similarity and MMR. Using three different types of queries, a keyword, conceptual, and technical one.

#Trials A & B

In [ ]:
keyword_query = "What were Abraham Lincoln's main points in the Second Inaugural Address?"
conceptual_query = "What were Martin Luther King Jr.'s hopes for equality in the 'I Have a Dream' speech?"
technical_query = "Which speech did Franklin D. Roosevelt give after Pearl Harbor and what date did he mention?"

queries = {
    "Keyword / Factual": keyword_query,
    "Conceptual": conceptual_query,
    "Technical": technical_query
}

def test_retriever(query, name, retriever):
    print(f"\n--- Testing: {name} ---")
    print(f"Query: {query}\n")

    # BM25 or Vector retrievers
    if hasattr(retriever, "get_relevant_documents"):
        docs = retriever.get_relevant_documents(query)
    elif hasattr(retriever, "vectorstore"):
        docs = retriever.vectorstore.similarity_search(query, k=5)
    else:
        raise ValueError(f"{name} retriever is not compatible with this test.")

    # Show top 3 results
    for i, doc in enumerate(docs[:3]):
        source = getattr(doc, "metadata", {}).get("source", "Unknown") if hasattr(doc, "metadata") else "Unknown"
        content = getattr(doc, "page_content", str(doc))[:200].replace("\n", " ")
        print(f"Rank {i+1}: [{source}] {content}...")


# Only Naive Vector and MMR retrievers
retrievers = {
    "Trial A: Naive Vector": naive_vector_retriever,
    "Trial B: MMR": mmr_retriever
}

# Run the tests
for qtype, qtext in queries.items():
    print(f"\n-------- {qtype} Query -------")
    for name, retriever in retrievers.items():
        test_retriever(qtext, name, retriever)


-------- Keyword / Factual Query -------

--- Testing: Trial A: Naive Vector ---
Query: What were Abraham Lincoln's main points in the Second Inaugural Address?

Rank 1: [second_inaugural.txt] Fellow countrymen:  At this second appearing to take the oath of the presidential office, there is less occasion for an extended address than there was at the first.  Then a statement, somewhat in det...
Rank 2: [second_inaugural.txt] With malice toward none; with charity for all; with firmness in the right, as God gives us to see the right, let us strive on to finish the work we are in; to bind up the nation's wounds; to care for ...
Rank 3: [second_inaugural.txt] On the occasion corresponding to this four years ago, all thoughts were anxiously directed to an impending civil war.  All dreaded it-- all sought to avert it.  While the inaugural address was being d...

--- Testing: Trial B: MMR ---
Query: What were Abraham Lincoln's main points in the Second Inaugural Address?

Rank 1: [second_inau

The chunks that were retrieved are relevant to the user query. But both Naive and MMR retrieval methods yielded the same chunks. This was likely due to my relatively small dataset. The queries were also very specific and no similar chunks were found for Naive to be redundant, so the results were naturally diverse anyway.

MMR only improves the results when there's multiple competing chunks. Within this dataset, each query maps cleanly to a single document, therefore the diversity did not improve the retrieval.

Testing chunk retrieval using Hybrid Search from the notebook.

#Trial C

In [ ]:
keyword_query = "What were Abraham Lincoln's main points in the Second Inaugural Address?"
conceptual_query = "What were Martin Luther King Jr.'s hopes for equality in the 'I Have a Dream' speech?"
technical_query = "Which speech did Franklin D. Roosevelt give after Pearl Harbor and what date did he mention?"

hybrid_queries = {
    "Keyword / Factual": keyword_query,
    "Conceptual": conceptual_query,
    "Technical": technical_query
}

def test_hybrid(query, name):
    print(f"\n--- Testing: {name} ---")
    print(f"Query: {query}\n")

    # Hybrid retriever uses .invoke() in the current setup
    docs = hybrid_retriever.invoke(query)

    # Show top 3 results
    for i, doc in enumerate(docs[:3]):
        source = doc.metadata.get("source", "Unknown")
        content = doc.page_content[:200].replace("\n", " ")
        print(f"Rank {i+1}: [{source}] {content}...")

# Run the hybrid tests
for qtype, qtext in hybrid_queries.items():
    test_hybrid(qtext, "Hybrid Retriever Test")


--- Testing: Hybrid Retriever Test ---
Query: What were Abraham Lincoln's main points in the Second Inaugural Address?

Rank 1: [second_inaugural.txt] On the occasion corresponding to this four years ago, all thoughts were anxiously directed to an impending civil war.  All dreaded it-- all sought to avert it.  While the inaugural address was being d...
Rank 2: [second_inaugural.txt] One-eighth of the whole population were colored slaves, not distributed generally over the Union, but localized in the Southern part of it. These slaves constituted a peculiar and powerful interest.  ...
Rank 3: [second_inaugural.txt] Fellow countrymen:  At this second appearing to take the oath of the presidential office, there is less occasion for an extended address than there was at the first.  Then a statement, somewhat in det...

--- Testing: Hybrid Retriever Test ---
Query: What were Martin Luther King Jr.'s hopes for equality in the 'I Have a Dream' speech?

Rank 1: [i_have_a_dream.txt] righteousne

Hybrid found chunks in the documents which had some keyword similarity, hence why a chunk from four_freedoms.txt is retrieved when the query is about Martin Luther King Jr.

Hybrid search incorporates keyword matching, but may introduce less relevant results when different documents share overlapping vocabulary.

# **The Generation Phase**
* The LLM now can now respond to the user with the additional chunk information from the documents.



#Trial A: Naive Similarity

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

try:
    from langchain_google_genai import ChatGoogleGenerativeAI
except ImportError:
    raise ImportError(
        "langchain_google_genai not installed. "
        "Install it with `pip install langchain-google-genai` or use a local LLM alternative."
    )

try:
    from langchain_classic import hub
except ImportError:
    import langchainhub as hub

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0
)

llm_with_retry = llm.with_retry(
    stop_after_attempt=5,
    wait_exponential_jitter=True
)

prompt = hub.pull("rlm/rag-prompt")

rag_chain = (
    RunnableParallel({
        "context": naive_vector_retriever,
        "question": RunnablePassthrough()
    })
    | prompt
    | llm_with_retry
    | StrOutputParser()
)

queries = [
    "What were Abraham Lincoln's main points in the Second Inaugural Address?",
    "What were Martin Luther King Jr.'s hopes for equality in the 'I Have a Dream' speech?",
    "Which speech did Franklin D. Roosevelt give after Pearl Harbor and what date did he mention?",
    "What did Franklin D. Roosevelt outline in the Four Freedoms speech?"
]

# Run the queries through the RAG chain
for query in queries:
    try:
        answer = rag_chain.invoke(query)
        print(f"QUESTION: {query}")
        print(f"\nANSWER:\n{answer}\n{'-'*80}\n")
    except Exception as e:
        print(f"RAG Chain failed for query: {query}\nError: {e}\n{'-'*80}\n")

QUESTION: What were Abraham Lincoln's main points in the Second Inaugural Address?

ANSWER:
In his Second Inaugural Address, Abraham Lincoln identified slavery as the cause of the Civil War and reflected on the shared responsibility and divine purpose behind the conflict. He acknowledged that both sides dreaded war but accepted it for their respective causes. Lincoln concluded with a powerful plea for reconciliation, charity, and a lasting peace, urging the nation to bind its wounds and care for those affected by the war.
--------------------------------------------------------------------------------

QUESTION: What were Martin Luther King Jr.'s hopes for equality in the 'I Have a Dream' speech?

ANSWER:
Martin Luther King Jr. hoped the nation would live out its creed that all men are created equal, with sons of former slaves and slave-owners sitting together in brotherhood. He dreamed his children would be judged by their character, not their skin color, and that states like Mississi

#Trial B: MMR

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

try:
    from langchain_google_genai import ChatGoogleGenerativeAI
except ImportError:
    raise ImportError(
        "langchain_google_genai not installed. "
        "Install it with `pip install langchain-google-genai` or use a local LLM alternative."
    )

try:
    from langchain_classic import hub
except ImportError:
    import langchainhub as hub

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0
)

llm_with_retry = llm.with_retry(
    stop_after_attempt=5,
    wait_exponential_jitter=True
)

prompt = hub.pull("rlm/rag-prompt")

rag_chain = (
    RunnableParallel({
        "context": mmr_retriever,
        "question": RunnablePassthrough()
    })
    | prompt
    | llm_with_retry
    | StrOutputParser()
)

queries = [
    "What were Abraham Lincoln's main points in the Second Inaugural Address?",
    "What were Martin Luther King Jr.'s hopes for equality in the 'I Have a Dream' speech?",
    "Which speech did Franklin D. Roosevelt give after Pearl Harbor and what date did he mention?",
    "What did Franklin D. Roosevelt outline in the Four Freedoms speech?"
]

# Run the queries through the RAG chain
for query in queries:
    try:
        answer = rag_chain.invoke(query)
        print(f"QUESTION: {query}")
        print(f"\nANSWER:\n{answer}\n{'-'*80}\n")
    except Exception as e:
        print(f"RAG Chain failed for query: {query}\nError: {e}\n{'-'*80}\n")

QUESTION: What were Abraham Lincoln's main points in the Second Inaugural Address?

ANSWER:
In his Second Inaugural Address, Abraham Lincoln reflected on the Civil War, noting that both sides dreaded it but one would make war to destroy the Union and the other would accept war to save it. He emphasized the need to finish the work with "malice toward none" and "charity for all." Lincoln's main points included binding up the nation's wounds, caring for veterans and their families, and striving to achieve a just and lasting peace.
--------------------------------------------------------------------------------

QUESTION: What were Martin Luther King Jr.'s hopes for equality in the 'I Have a Dream' speech?

ANSWER:
In his "I Have a Dream" speech, Martin Luther King Jr. hoped that the nation would live out the true meaning of its creed, that all men are created equal. He envisioned a future where sons of former slaves and slave-owners could sit together at the table of brotherhood. King als

#Trial C: Hybrid

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Hub import for modular LangChain
try:
    from langchain_classic import hub
except ImportError:
    import langchainhub as hub


# Gemini 2.5 Flash is the 'sweet spot' for speed and availability.
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0
)



# Attach Rate-Limit Protection
llm_with_retry = llm.with_retry(
    stop_after_attempt=5,
    wait_exponential_jitter=True
)



# Pull the standard RAG prompt
prompt = hub.pull("rlm/rag-prompt")



# THE LCEL PIPELINE (The "Pipe" Syntax)
rag_chain = (
    RunnableParallel({
        "context": hybrid_retriever,
        "question": RunnablePassthrough()
    })
    | prompt
    | llm_with_retry
    | StrOutputParser()
)



# Final Execution
queries = [
    "What were Abraham Lincoln's main points in the Second Inaugural Address?",
    "What were Martin Luther King Jr.'s hopes for equality in the 'I Have a Dream' speech?",
    "Which speech did Franklin D. Roosevelt give after Pearl Harbor and what date did he mention?",
    "What did Franklin D. Roosevelt outline in the Four Freedoms speech?"
]

print("--- Final RAG Project Test (Gemini 2.5 Flash) ---\n")

for query in queries:
    try:
        answer = rag_chain.invoke(query)
        print(f"QUESTION: {query}")
        print(f"\nANSWER:\n{answer}\n{'-'*80}\n")
    except Exception as e:
        print(f"RAG Chain failed for query: {query}\nError: {e}\n{'-'*80}\n")


--- Final RAG Project Test (Gemini 2.5 Flash) ---

QUESTION: What were Abraham Lincoln's main points in the Second Inaugural Address?

ANSWER:
In his Second Inaugural Address, Abraham Lincoln reflected on the Civil War's origins, stating that both sides dreaded it but one sought to dissolve the Union over slavery, which he identified as the war's cause. He pondered God's purpose in the conflict, suggesting it was to remove the offense of slavery. Finally, Lincoln called for reconciliation, urging the nation to act "with malice toward none, with charity for all" to achieve a just and lasting peace.
--------------------------------------------------------------------------------

QUESTION: What were Martin Luther King Jr.'s hopes for equality in the 'I Have a Dream' speech?

ANSWER:
Martin Luther King Jr. dreamed that the nation would live out the true meaning of its creed, that "all men are created equal," and that sons of former slaves and slave-owners would sit together at the table o

#Analysis

1. Trial A and B both returned a variety of chunks. This was not a typical result as Naive tends to produce redundant results. But in this case redundancy did not occur because the dataset is small and structured well. Each query can map cleanly to a single document. I expect that with a larger dataset and similar information across the documents. Naive would produce redundant chunks, while MMR would likely include more diverse chunks.

2. The Hybrid retriever combined semantic and keyword search so it recieved additional chunks which did not appear in trials A and B. But these additional results were not always relevant to the query. For example, keyword overlap about freedom caused the Hybrid search to pull chunks from the Four Freedoms speech. While the query was asking about the I Have a Dream speech.

3. The LLM was grounded in all three of the trials. It did not seem to guess or rely on training knowledge. All responses were supported by the retrieved chunks. This means that the retrieval quality had a direct impact on groundedness. Since the chunks were high in context the LLM generated accurate responses with no hallucination.

4. Based on my dataset, surprisingly the Naive Similarity is the most effective retrieval strategy for production. It showed relevant and precise results, and did not introduce unrelated chunks like in Hybrid. MMR failed to improve performance due to the lack of overlapping documents. If I were to increase my dataset, I would opt for MMR or Hybrid instead to handle introduced diversity, and the handling of keyword specific queries.

#Response Quality Scores

| Question Type | Naive Similarity | MMR | Hybrid |
|----------|----------|----------|----------|
| Factual    | 5     | 5     | 5 |
| Conceptual  | 5     | 5     | 5 |
|  Technical   | 5     | 5     | 5

All three of the methods recieved a high score since the dataset is very structured and focused. Each query was able to map to the specific document. Retrieval was consistently accurate for all three strategies.

Naive method performed slightly better in terms of precision. While Hybrid showed slightly more diversity but with slightly less relevance. MMR did not differ from Naive due to the limited need for diversity in the dataset.

#Reranker Step (Extra Credit)

This step takes the top results and uses a more powerful but slower model to calculate a more accurate relevance score.

In [6]:
# 2026 UPDATED IMPORTS
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Hub import for modular LangChain
try:
    from langchain_classic import hub
except ImportError:
    import langchainhub as hub

# Reranker imports
try:
    from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
    from langchain.retrievers.document_compressors.flashrank_rerank import FlashrankRerank
except ImportError:
    from langchain_classic.retrievers import ContextualCompressionRetriever
    from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank

# Initialize LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0
)

# Attach Rate-Limit Protection
llm_with_retry = llm.with_retry(
    stop_after_attempt=5,
    wait_exponential_jitter=True
)

# Pull the standard RAG prompt
prompt = hub.pull("rlm/rag-prompt")

# Initialize FlashRank Reranker
compressor = FlashrankRerank(
    model="ms-marco-MiniLM-L-12-v2",
    top_n=3
)

# Wrap the hybrid retriever with contextual compression
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriever
)

# RAG Pipeline using Reranker
rag_chain_reranked = (
    RunnableParallel({
        "context": compression_retriever,
        "question": RunnablePassthrough()
    })
    | prompt
    | llm_with_retry
    | StrOutputParser()
)

# Difficult Query Test
query = "Which speech did Franklin D. Roosevelt give after Pearl Harbor and what date did he mention?"

print("--- Running RAG pipeline with reranker ---\n")
answer = rag_chain_reranked.invoke(query)
print(f"QUESTION: {query}\n")
print(f"ANSWER:\n{answer}\n{'-'*80}\n")

# Inspect the top ranked chunks
final_docs = compression_retriever.invoke(query)
print(f"Retrieved {len(final_docs)} chunks (top {compressor.top_n} after rerank):\n")

for i, doc in enumerate(final_docs):
    source = doc.metadata.get("source", "Unknown Source")
    print(f"RANK {i+1} (Source: {source}):\n{doc.page_content[:450]}...\n{'-'*60}")

ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:01<00:00, 22.1MiB/s]


--- Running RAG pipeline with reranker ---

QUESTION: Which speech did Franklin D. Roosevelt give after Pearl Harbor and what date did he mention?

ANSWER:
Franklin D. Roosevelt gave the "Day of Infamy" speech after the attack on Pearl Harbor. In this speech, he mentioned December 7th, 1941, as the date of the attack. He famously referred to it as "a date which will live in infamy."
--------------------------------------------------------------------------------

Retrieved 3 chunks (top 3 after rerank):

RANK 1 (Source: day_of_infamy.txt):
Mr. Vice President, Mr. Speaker, Members of the Senate, and of the House
of Representatives:

Yesterday, December 7th, 1941 -- a date which will live in infamy -- the
United States of America was suddenly and deliberately attacked by naval
and air forces of the Empire of Japan.

The United States was at peace with that nation and, at the solicitation
of Japan, was still in conversation with its government and its emperor
looking toward the mainte...


#Rerank Evaluation:

 Did the Reranker move a more relevant chunk from the "bottom" of the list to the "top"?
*   Yes, the reranker moved more relevant chunks to the top. This allows the LLM to see the most important context first. Before the reranking with the hybrid approach, at rank #2 there was a chunk from a less relavant document. The reranking fixed this by moving a more relevant chunk into the appropriate ranking.

Is the extra latency, i.e., time taken to rerank, worth the improvement in precision for your specific use case?
*   Yes, the extra latency is worth it for my use case. FlashRank reranking adds an extra model pass over the chunks which were received. In my case this only adds an additional few seconds of latency. Therefore the precision gain and relevance boost is definitely worth it.

